# Symbolic Music Generation — RNN vs GRU vs LSTM

This notebook trains three sequence generators on the same symbolic melody dataset:
- Simple RNN
- GRU
- LSTM

The goal is the same as the character-level notebook: keep the pipeline readable and compare how each recurrent architecture learns sequential structure.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "nirma_university_language_models").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [ ]:
import matplotlib.pyplot as plt

from nirma_university_language_models import (
    MusicModel,
    build_music_dataloaders,
    build_music_vocabulary,
    encode_music_tokens,
    flatten_music_token_sequences,
    get_device,
    load_music_token_sequences,
    melody_to_text,
    melody_tokens_to_waveform,
    sample_music_model,
    save_waveform_as_wav,
    seed_everything,
    train_music_model,
)


In [ ]:
device = get_device()
SEED = 42
seed_everything(SEED)
print("Using device:", device)

melodies, data_path = load_music_token_sequences()
vocab, token_to_idx, idx_to_token = build_music_vocabulary(melodies)
flat_tokens = flatten_music_token_sequences(melodies)
encoded = encode_music_tokens(flat_tokens, token_to_idx)

print("Dataset path:", data_path)
print("Melodies:", len(melodies))
print("Vocabulary size:", len(vocab))
print("Flat token count:", len(flat_tokens))

## 1. Build training windows

Each training example is a fixed-length token window.
The target is the same window shifted by one step, so the model learns next-token prediction.

In [ ]:
SEQ_LEN = 12
BATCH_SIZE = 16
train_loader, val_loader = build_music_dataloaders(
    encoded,
    seq_len=SEQ_LEN,
    batch_size=BATCH_SIZE,
)

print("Train windows:", len(train_loader.dataset))
print("Validation windows:", len(val_loader.dataset))

## 2. Preview the model

Only the recurrent layer changes between RNN, GRU, and LSTM.
The embedding and output projection stay the same.

In [ ]:
MODEL_KWARGS = {"embed_dim": 32, "hidden_size": 96}
preview_model = MusicModel(len(vocab), rnn_type="RNN", **MODEL_KWARGS)
preview_model

## 3. Train the three models

This keeps the optimizer, learning rate, dataset, and sequence length fixed so the architecture comparison is fair.

In [ ]:
EPOCHS = 12

rnn_model, rnn_train_losses, rnn_val_losses = train_music_model(
    "RNN", len(vocab), train_loader, val_loader, device, epochs=EPOCHS, **MODEL_KWARGS
)
gru_model, gru_train_losses, gru_val_losses = train_music_model(
    "GRU", len(vocab), train_loader, val_loader, device, epochs=EPOCHS, **MODEL_KWARGS
)
lstm_model, lstm_train_losses, lstm_val_losses = train_music_model(
    "LSTM", len(vocab), train_loader, val_loader, device, epochs=EPOCHS, **MODEL_KWARGS
)

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(rnn_val_losses, label="RNN validation")
plt.plot(gru_val_losses, label="GRU validation")
plt.plot(lstm_val_losses, label="LSTM validation")
plt.title("Validation Loss Comparison")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Sample generated melodies

Try changing the seed motif or the temperature to show how generation quality and diversity change.

In [ ]:
seed_motif = ["C4_q", "E4_q", "G4_q"]

rnn_sample = sample_music_model(rnn_model, token_to_idx, idx_to_token, device, start_tokens=seed_motif)
gru_sample = sample_music_model(gru_model, token_to_idx, idx_to_token, device, start_tokens=seed_motif)
lstm_sample = sample_music_model(lstm_model, token_to_idx, idx_to_token, device, start_tokens=seed_motif)

print("RNN :", melody_to_text(rnn_sample))
print("GRU :", melody_to_text(gru_sample))
print("LSTM:", melody_to_text(lstm_sample))

In [ ]:
output_dir = PROJECT_ROOT / "src" / "music_generation" / "generated_samples"
waveform = melody_tokens_to_waveform(lstm_sample, tempo_bpm=120)
wav_path = save_waveform_as_wav(waveform, output_dir / "lstm_sample.wav")
print("Saved generated sample to:", wav_path)